# Exercise 9 — Beer Delivery


## Task

A Berlin trading company delivers beer using a truck with five compartments. Each compartment has a fixed capacity and can carry **at most one beer type**.

### Compartment capacities

| Compartment | Capacity (liters) |
|---|---:|
| 1 | 2700 |
| 2 | 2800 |
| 3 | 1100 |
| 4 | 1800 |
| 5 | 3400 |

The company must serve an order for three beer brands. Unmet demand incurs a per-liter penalty, but shortage is limited for each brand. Quantities are counted in full liters.

| Beer | Demand (liters) | Penalty per liter | Maximum shortage |
|---|---:|---:|---:|
| Berliner Pilsner | 2900 | 10 | 500 |
| Berliner Kindl | 4000 | 8 | 500 |
| Schultheiss | 4900 | 6 | 300 |

Tasks:

1. Formulate the problem algebraically as a mixed-integer model.
2. Translate it into JuMP and solve it.
3. Interpret the compartment assignment and any remaining shortage.


## Reference implementation


In [ ]:
using HiGHS, JuMP

compartments = collect(1:5) # compartments
brands = ["pilsener", "kindel", "schultheiss"] # brands

s = Dict(zip(compartments, [2700, 2800, 1100, 1800, 3400]))
d = Dict(zip(brands, [2900, 4000, 4900]))
ms = Dict(zip(brands, [500, 500, 300]))
p = Dict(zip(brands, [10, 8, 6]))

m = Model(HiGHS.Optimizer)
@variable(m, x[compartments, brands], Bin)
@variable(m, q[brands] >= 0, Int) # penalty variable for brands
@constraint(
    m,
    market_clearing[b=brands],
    sum(x[c, b] * s[c] for c in compartments) >= d[b] - ms[b]
)
@constraint(m, penalty[b=brands], q[b] >= d[b] - sum(x[c, b]*s[c] for c in compartments))
@constraint(m, compartments[c=compartments], sum(x[c, b] for b in brands) <= 1)
@objective(m, Min, sum(p[b]*q[b] for b in brands))
optimize!(m)
objective_value(m)

value.(x)
value.(q)
